# Grid Search — seis estimadores campeões (012 + polinomial grau 2)

**Objetivo:** realizar a **otimização de hiperparâmetros por Grid Search** dos seis modelos escolhidos após os 18 experimentos exploratórios. Este notebook **não repete o AutoML** e **não introduz ruído**.

| Alvo | Modelo de referência | Representação |
|---|---|---|
| `R0` | Bayesian Ridge (`br`) | `SEQ_012_POLI2` |
| `R1` | Ridge (`ridge`) | `SEQ_012_POLI2` |
| `L0` | CatBoost (`catboost`) | `SEQ_012_POLI2` |
| `L1` | Ridge (`ridge`) | `SEQ_012_POLI2` |
| `C0` | Bayesian Ridge (`br`) | `SEQ_012_POLI2` |
| `C1` | Bayesian Ridge (`br`) | `SEQ_012_POLI2` |

**Desenho experimental preservado:** 29.419 cenários MATLAB sem ruído; 80% treino / 20% teste com semente 42; cinco dobras de CV; fasores ABC → componentes de Fortescue 012; 24 atributos de módulo/ângulo em graus; polinômios de grau 2 **no pipeline do PyCaret**; normalização z-score; cada parâmetro possui um regressor independente.

**Critério de seleção:** menor **MAPE na validação cruzada**. O teste reservado é usado **somente após** escolher entre o modelo padrão e o otimizado. MAPE, MAE, RMSE e R² são calculados nas unidades físicas originais. Os CSVs de todas as combinações da busca são salvos para documentação da dissertação.

**Atenção:** o conjunto de teste já foi inspecionado nas rodadas exploratórias anteriores. Por isso, as métricas aqui devem ser interpretadas como confirmação no teste reservado existente, não como uma nova validação totalmente independente. Não usaremos seus resultados para escolher novos hiperparâmetros.

## 0. Configuração

Coloque `Analise_PN_FP_5_Cenarios.csv` na mesma pasta deste notebook. Ambiente recomendado: Python 3.10/3.11 e PyCaret 3.3.2, com CatBoost instalado. Os arquivos gerados vão para uma **nova pasta**, sem sobrescrever os rankings anteriores.

Se necessário, você pode usar `ALVOS_EXECUTAR = ['R1']` para experimentar um parâmetro antes dos seis. `PULAR_CONCLUIDOS=True` só ignora resultados totalmente concluídos desta mesma versão de notebook.

In [ ]:
from pathlib import Path
from itertools import product
from math import prod
import json
import time
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ARQUIVO_BASE = Path('Analise_PN_FP_5_Cenarios.csv')
PASTA_SAIDA = Path('resultados_GridSearch_6_Campeoes_012_POLI2')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

SEMENTE_SPLIT = 42
FRACAO_TESTE = 0.20
DOBRAS_CV = 5
N_JOBS = 2                     # -1 para usar mais CPUs, observando a RAM
ALVOS = ['R0', 'R1', 'L0', 'L1', 'C0', 'C1']
ALVOS_EXECUTAR = list(ALVOS)    # ou ['R1'] para executar só esse alvo
PULAR_CONCLUIDOS = True
SALVAR_MODELOS = True
SALVAR_PREDICOES = False
PLOTAR_DISPERSOES = True
AMOSTRAS_DISPERSAO = 3000

CAMPEOES = {
    'R0': 'br', 'R1': 'ridge', 'L0': 'catboost',
    'L1': 'ridge', 'C0': 'br', 'C1': 'br',
}

# Grades pequenas, explícitas e exaustivas; podem ser ampliadas numa rodada futura.
# Incluímos o valor padrão 1e-6 em cada parâmetro do Bayesian Ridge.
GRADES = {
    'ridge': {
        'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    'br': {
        'alpha_1': [1e-7, 1e-6],
        'alpha_2': [1e-7, 1e-6],
        'lambda_1': [1e-7, 1e-6],
        'lambda_2': [1e-7, 1e-6],
    },
    'catboost': {
        'depth': [4, 6],
        'learning_rate': [0.03, 0.10],
        'iterations': [300, 600],
        'l2_leaf_reg': [3, 7],
    },
}

assert set(ALVOS_EXECUTAR).issubset(ALVOS)
for alvo in ALVOS_EXECUTAR:
    modelo = CAMPEOES[alvo]
    qtd = prod(len(v) for v in GRADES[modelo].values())
    print(f'{alvo}: {modelo} | {qtd} combinações × {DOBRAS_CV} dobras = {qtd*DOBRAS_CV} ajustes na busca')
print('Python:', sys.version.split()[0])

## 1. Leitura da base e mesma divisão original

**Nenhum metadado de fator de potência ou `ID_Cenario` entra como característica.** O particionamento usa a mesma chamada `train_test_split`, mesma semente e índices ordenados do notebook dos 18 experimentos.

In [ ]:
FASORES_ABC = [
    'VA_in', 'VB_in', 'VC_in', 'IA_in', 'IB_in', 'IC_in',
    'VA_out', 'VB_out', 'VC_out', 'IA_out', 'IB_out', 'IC_out'
]
CARACTERISTICAS_ABC = [f'{f}_{s}' for f in FASORES_ABC for s in ('mod', 'phase')]
METADADOS = ['Fator_Potencia', 'Tipo_FP']

if not ARQUIVO_BASE.exists():
    raise FileNotFoundError(f'CSV MATLAB não encontrado: {ARQUIVO_BASE.resolve()}')
orig = pd.read_csv(ARQUIVO_BASE).reset_index(drop=True)
exigidas = CARACTERISTICAS_ABC + ALVOS + METADADOS
faltantes = sorted(set(exigidas) - set(orig.columns))
assert not faltantes, f'Colunas ausentes: {faltantes}'
orig = orig[exigidas].copy()
numericas = CARACTERISTICAS_ABC + ALVOS + ['Fator_Potencia']
orig[numericas] = orig[numericas].apply(pd.to_numeric, errors='raise')
assert not orig[numericas].isna().any().any()
assert np.isfinite(orig[numericas].to_numpy(dtype=float)).all()
assert (orig[ALVOS].to_numpy() > 0).all()
assert (orig[[f'{f}_mod' for f in FASORES_ABC]].to_numpy() > 0).all()

orig['ID_Cenario'] = np.arange(len(orig), dtype=np.int64)
idx_treino, idx_teste = train_test_split(
    np.arange(len(orig)), test_size=FRACAO_TESTE,
    random_state=SEMENTE_SPLIT, shuffle=True
)
idx_treino = np.sort(idx_treino)
idx_teste = np.sort(idx_teste)
assert not np.intersect1d(idx_treino, idx_teste).size
assert len(idx_treino) + len(idx_teste) == len(orig)

pd.DataFrame({
    'ID_Cenario': orig['ID_Cenario'],
    'Particao': np.where(orig['ID_Cenario'].isin(idx_treino), 'Treino', 'Teste')
}).to_csv(PASTA_SAIDA / 'particao_80_20.csv', index=False)
print(f'Cenários: {len(orig):,} | Treino: {len(idx_treino):,} | Teste: {len(idx_teste):,}')

## 2. Transformação ABC → 012 (Fortescue)

Fasores reconstruídos a partir de módulos e ângulos **em graus**. A versão 012 tem 24 atributos. Os atributos quadráticos e de interação são produzidos dentro do `setup()`.

\[
\begin{bmatrix}X_0\\X_1\\X_2\end{bmatrix}
= \frac13 \begin{bmatrix}1&1&1\\1&a&a^2\\1&a^2&a\end{bmatrix}
\begin{bmatrix}X_A\\X_B\\X_C\end{bmatrix},\quad a=e^{j2\pi/3}.
\]

In [ ]:
a = np.exp(2j * np.pi / 3)

def fortescue_012(Xabc: np.ndarray) -> np.ndarray:
    matriz = np.array([[1,1,1], [1,a,a**2], [1,a**2,a]], dtype=np.complex128) / 3
    return Xabc @ matriz.T

def fortescue_inversa(X012: np.ndarray) -> np.ndarray:
    matriz_inv = np.array([[1,1,1], [1,a**2,a], [1,a,a**2]], dtype=np.complex128)
    return X012 @ matriz_inv.T

def abc_complexo(df: pd.DataFrame, tipo: str, extremo: str) -> np.ndarray:
    trio = []
    for fase in 'ABC':
        nome = f'{tipo}{fase}_{extremo}'
        modulo = df[f'{nome}_mod'].to_numpy(dtype=float)
        angulo = df[f'{nome}_phase'].to_numpy(dtype=float)
        trio.append(modulo * np.exp(1j * np.deg2rad(angulo)))
    return np.stack(trio, axis=1)

assert np.allclose(fortescue_012(np.array([[1+0j, a**2, a]])), [[0,1,0]], atol=1e-12)
colunas_seq = {}
for extremo in ('in', 'out'):
    for tipo in ('V', 'I'):
        Xabc = abc_complexo(orig, tipo, extremo)
        X012 = fortescue_012(Xabc)
        assert np.allclose(fortescue_inversa(X012), Xabc, rtol=1e-10, atol=1e-8)
        for seq in range(3):
            nome = f'{tipo}{seq}_{extremo}'
            colunas_seq[f'{nome}_mod'] = np.abs(X012[:,seq])
            colunas_seq[f'{nome}_phase'] = np.rad2deg(np.angle(X012[:,seq]))

base = pd.DataFrame(colunas_seq)
CARACTERISTICAS_012 = list(base.columns)
assert len(CARACTERISTICAS_012) == 24
base[ALVOS] = orig[ALVOS].copy()
base['ID_Cenario'] = orig['ID_Cenario']
assert np.array_equal(base[ALVOS].to_numpy(), orig[ALVOS].to_numpy())
assert np.isfinite(base[CARACTERISTICAS_012].to_numpy(dtype=float)).all()
print('Base 012 validada: 24 atributos; todos os alvos preservados.')

## 3. Grid Search com PyCaret (cinco dobras)

Para cada parâmetro:

1. `create_model()` ajusta o **modelo de referência** (hiperparâmetros padrão) e calcula a CV.
2. `tune_model(..., search_algorithm='grid', custom_grid=..., return_tuner=True)` avalia **todas** as combinações da grade especificada.
3. Exportamos a tabela integral `cv_results_`, a CV do modelo original e a CV do modelo ajustado, e os melhores hiperparâmetros.
4. Selecionamos **exclusivamente pelo MAPE da CV** entre a versão padrão e a otimizada; em empate, fica o padrão.
5. **Somente após essa escolha**, calculamos as métricas no mesmo teste reservado e salvamos o pipeline escolhido sem `finalize_model()`.

Pode demorar, principalmente para CatBoost. Os resultados individuais são salvos assim que cada parâmetro termina. Se uma busca falhar, o notebook registra um arquivo de erro e interrompe a célula (não substitui silenciosamente o resultado).

In [ ]:
try:
    import pycaret
    from pycaret.regression import RegressionExperiment
except ImportError as exc:
    raise ImportError('Instale PyCaret 3.3.2 em Python 3.10 ou 3.11.') from exc
print('PyCaret:', pycaret.__version__)
assert pycaret.__version__.split('.')[0] == '3'

def converter_json(x):
    if isinstance(x, (np.generic,)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    return str(x)

def resumo_cv(tabela: pd.DataFrame) -> dict:
    # Extrai linha Mean da CV do PyCaret independentemente da posição.
    t = tabela.copy()
    nomes_indice = t.index.astype(str).str.lower().str.strip()
    if (nomes_indice == 'mean').any():
        linha = t.iloc[np.flatnonzero(nomes_indice == 'mean')[0]]
    elif 'Fold' in t.columns and t['Fold'].astype(str).str.lower().eq('mean').any():
        linha = t.loc[t['Fold'].astype(str).str.lower().eq('mean')].iloc[0]
    else:
        raise ValueError('Não encontrei a linha Mean na tabela CV: ' + str(t.index.tolist()))
    necessarias = ['MAPE', 'RMSE', 'R2', 'MAE']
    faltantes = [m for m in necessarias if m not in linha.index]
    if faltantes:
        raise ValueError('Faltam métricas CV: ' + str(faltantes))
    return {
        'MAPE_CV_%': 100 * float(linha['MAPE']),
        'RMSE_CV': float(linha['RMSE']),
        'R2_CV': float(linha['R2']),
        'MAE_CV': float(linha['MAE']),
    }

def metricas_teste(real, previsto) -> dict:
    y = np.asarray(real, dtype=float)
    p = np.asarray(previsto, dtype=float)
    assert np.isfinite(y).all() and np.isfinite(p).all() and np.all(y != 0)
    return {
        'MAPE_teste_%': 100 * float(np.mean(np.abs((y-p)/y))),
        'MAE_teste': float(mean_absolute_error(y, p)),
        'RMSE_teste': float(np.sqrt(mean_squared_error(y,p))),
        'R2_teste': float(r2_score(y,p)),
    }

def avaliar(exp, modelo, teste, colunas_X, alvo):
    previsoes = exp.predict_model(
        modelo, data=teste[colunas_X].copy(), round=15, verbose=False
    )
    coluna = 'prediction_label' if 'prediction_label' in previsoes.columns else 'Label'
    assert coluna in previsoes.columns, previsoes.columns.tolist()
    y = teste[alvo].to_numpy(dtype=float)
    p = previsoes[coluna].to_numpy(dtype=float)
    assert len(y) == len(p) == len(idx_teste)
    return metricas_teste(y, p), y, p

resumo_em_memoria = []
for alvo in ALVOS_EXECUTAR:
    modelo_id = CAMPEOES[alvo]
    grade = GRADES[modelo_id]
    arq_marcador = PASTA_SAIDA / f'CONCLUIDO_{alvo}.json'
    arq_resultado = PASTA_SAIDA / f'resultado_grid_{alvo}.csv'
    if PULAR_CONCLUIDOS and arq_marcador.exists() and arq_resultado.exists():
        print(f'[{alvo}] Já concluído nesta versão; leitura do resultado salvo.')
        resumo_em_memoria.append(pd.read_csv(arq_resultado).iloc[0].to_dict())
        continue

    print(f'\n===== {alvo} | {modelo_id} | SEQ_012_POLI2 | Grid Search =====', flush=True)
    t0 = time.perf_counter()
    cols = CARACTERISTICAS_012 + [alvo]
    treino = base.iloc[idx_treino][cols].reset_index(drop=True)
    teste = base.iloc[idx_teste][cols].reset_index(drop=True)

    exp = RegressionExperiment()
    exp.setup(
        data=treino,
        target=alvo,
        test_data=teste,
        session_id=SEMENTE_SPLIT,
        normalize=True,
        normalize_method='zscore',
        polynomial_features=True,
        polynomial_degree=2,
        fold_strategy='kfold',
        fold=DOBRAS_CV,
        fold_shuffle=True,
        index=False,
        html=False,
        verbose=False,
        n_jobs=N_JOBS,
    )

    try:
        original = exp.create_model(modelo_id, fold=DOBRAS_CV, round=10, verbose=False)
        cv_original = exp.pull().copy()
        cv_original.to_csv(PASTA_SAIDA / f'CV_original_{alvo}.csv')
        m_original = resumo_cv(cv_original)
        print('CV padrão:', m_original, flush=True)

        ajustado, tuner = exp.tune_model(
            original,
            fold=DOBRAS_CV,
            round=10,
            optimize='MAPE',
            custom_grid=grade,
            search_library='scikit-learn',
            search_algorithm='grid',
            choose_better=False,  # comparar explicitamente pela mesma métrica CV
            return_tuner=True,
            verbose=False,
        )
        cv_ajustado = exp.pull().copy()
        cv_ajustado.to_csv(PASTA_SAIDA / f'CV_otimizado_{alvo}.csv')
        m_ajustado = resumo_cv(cv_ajustado)
        print('CV otimizado:', m_ajustado, flush=True)

        if not hasattr(tuner, 'cv_results_'):
            raise RuntimeError('O retorno do tuning não expõe cv_results_; não há auditoria da grade.')
        pd.DataFrame(tuner.cv_results_).to_csv(
            PASTA_SAIDA / f'grid_completo_{alvo}.csv', index=False
        )
        melhores_hiperparametros = getattr(tuner, 'best_params_', {})
        with open(PASTA_SAIDA / f'melhores_hiperparametros_{alvo}.json', 'w', encoding='utf-8') as f:
            json.dump(melhores_hiperparametros, f, ensure_ascii=False, indent=2, default=converter_json)

        # A decisão acontece ANTES de acessar o teste.
        melhorou = m_ajustado['MAPE_CV_%'] < m_original['MAPE_CV_%']
        selecionado = ajustado if melhorou else original
        versao = 'GridSearch' if melhorou else 'Padrao'
        metrics_cv = m_ajustado if melhorou else m_original
        print(f'Selecionado pela CV: {versao}', flush=True)

        # Avaliação no mesmo teste reservado, apenas do selecionado.
        m_teste, y, p = avaliar(exp, selecionado, teste, CARACTERISTICAS_012, alvo)
        linha = {
            'Alvo': alvo,
            'Algoritmo_ID': modelo_id,
            'Algoritmo': type(selecionado).__name__,
            'Representacao': 'SEQ_012_POLI2',
            'Versao_escolhida': versao,
            'N_treino': len(treino), 'N_teste': len(teste),
            'Num_combinacoes_grade': prod(len(v) for v in grade.values()),
            'MAPE_CV_original_%': m_original['MAPE_CV_%'],
            'MAPE_CV_grid_%': m_ajustado['MAPE_CV_%'],
            'MAPE_CV_selecionado_%': metrics_cv['MAPE_CV_%'],
            'R2_CV_original': m_original['R2_CV'],
            'R2_CV_grid': m_ajustado['R2_CV'],
            'R2_CV_selecionado': metrics_cv['R2_CV'],
            'RMSE_CV_original': m_original['RMSE_CV'],
            'RMSE_CV_grid': m_ajustado['RMSE_CV'],
            'RMSE_CV_selecionado': metrics_cv['RMSE_CV'],
            **m_teste,
            'Tempo_total_segundos': time.perf_counter()-t0,
        }
        pd.DataFrame([linha]).to_csv(arq_resultado, index=False, float_format='%.15g')
        resumo_em_memoria.append(linha)

        if SALVAR_MODELOS:
            exp.save_model(
                selecionado, str(PASTA_SAIDA / f'modelo_GRID_{alvo}_{versao}'),
                verbose=False
            )
        if SALVAR_PREDICOES:
            pd.DataFrame({
                'ID_Cenario': base.iloc[idx_teste]['ID_Cenario'].to_numpy(),
                'Real': y, 'Predito': p,
            }).to_csv(PASTA_SAIDA / f'predicoes_{alvo}.csv', index=False, float_format='%.15g')

        if PLOTAR_DISPERSOES:
            rng_plot = np.random.default_rng(SEMENTE_SPLIT)
            idx_plot = rng_plot.choice(len(y), size=min(AMOSTRAS_DISPERSAO, len(y)), replace=False)
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.scatter(y[idx_plot], p[idx_plot], s=7, alpha=0.25)
            lmin = min(float(np.min(y)), float(np.min(p)))
            lmax = max(float(np.max(y)), float(np.max(p)))
            ax.plot([lmin,lmax], [lmin,lmax], '--', linewidth=1.1, color='black', label='Ideal')
            ax.set(xlabel=f'{alvo} real', ylabel=f'{alvo} estimado',
                   title=f'{alvo} — {modelo_id} ({versao}) | CV polinomial 012')
            ax.grid(alpha=0.2)
            ax.legend()
            fig.tight_layout()
            fig.savefig(PASTA_SAIDA / f'dispersao_{alvo}.png', dpi=150)
            plt.close(fig)

        # Marcador escrito por último; permite retomar execução interrompida.
        with arq_marcador.open('w', encoding='utf-8') as f:
            json.dump({'alvo':alvo,'versao':versao,
                       'arquivo_resultado':arq_resultado.name},f,indent=2)
        print(f'[{alvo}] Concluído. Teste MAPE={m_teste["MAPE_teste_%"]:.5f}% '
              f'R²={m_teste["R2_teste"]:.6f}', flush=True)
    except Exception as exc:
        with (PASTA_SAIDA / f'ERRO_{alvo}.txt').open('w', encoding='utf-8') as f:
            f.write(f'{type(exc).__name__}: {exc}\n')
        raise

print('\nEtapa de Grid Search encerrada para os alvos selecionados.')

## 4. Tabela consolidada para a dissertação

A tabela é construída a partir dos arquivos concluídos. Ela permite executar um parâmetro por vez sem repetir o que já terminou. **Os valores nesta célula só ficam disponíveis após executar o Grid Search no seu ambiente PyCaret.**

In [ ]:
arquivos = [PASTA_SAIDA / f'resultado_grid_{alvo}.csv' for alvo in ALVOS]
concluidos = [p for p in arquivos if p.exists() and
              (PASTA_SAIDA / f'CONCLUIDO_{p.stem.removeprefix("resultado_grid_")}.json').exists()]
if not concluidos:
    print('Nenhum Grid Search concluído ainda.')
else:
    resumo = pd.concat([pd.read_csv(p) for p in concluidos], ignore_index=True)
    resumo['Alvo'] = pd.Categorical(resumo['Alvo'], categories=ALVOS, ordered=True)
    resumo = resumo.sort_values('Alvo')
    resumo.to_csv(PASTA_SAIDA / 'RESUMO_GRID_SEARCH_SEIS_PARAMETROS.csv',
                  index=False, float_format='%.15g')
    cols = ['Alvo','Algoritmo','Versao_escolhida','MAPE_CV_original_%',
            'MAPE_CV_grid_%','MAPE_CV_selecionado_%','R2_CV_selecionado',
            'MAPE_teste_%','RMSE_teste','R2_teste','Num_combinacoes_grade']
    display(resumo[cols].round(6))
    print('Concluídos:', len(resumo), '/ 6. Saída:', PASTA_SAIDA.resolve())

## 5. Próxima etapa (não executada)

Depois de analisar as grades e a estabilidade na validação cruzada, os modelos escolhidos poderão ser usados na etapa posterior de **sensibilidade ao ruído**, mantendo separados os protocolos experimentais com e sem ruído. **Nenhuma perturbação fasorial é gerada neste notebook.**

**Importante:** `save_model()` salva o pipeline treinado sobre os dados de treinamento do experimento; não executamos `finalize_model()` para evitar incluir o teste reservado antes de concluir a avaliação.